In [18]:
import numpy as np
import random
from graphviz import Digraph

In [19]:
class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data - other.data, (self, other), '-')
        def _backward():
            self.grad += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, -other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += _unbroadcast(other.data * out.grad, self.data.shape)
            other.grad += _unbroadcast(self.data  * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            # d(loss)/dA = d(loss)/dC @ B.T
            # d(loss)/dB = A.T @ d(loss)/dC
            self.grad  += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float))
        out = Tensor(self.data ** exponent, (self,), f'**{exponent}')
        def _backward():
            self.grad += exponent * (self.data ** (exponent - 1)) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        """Reduce all elements to a scalar."""
        out = Tensor(self.data.sum(), (self,), 'sum')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        out = Tensor(np.tanh(self.data), (self,), 'Tanh')
        def _backward():
            self.grad += (1 - out.data**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other):    return self + other
    def __rmul__(self, other):    return self * other
    def __rsub__(self, other):    return Tensor(other) + (-self)

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)   # seed
        for node in reversed(topo):
            node._backward()

    def _unbroadcast(grad, shape):
        while grad.ndim > len(shape):
            grad = grad.sum(axis=0)
        for i, (g, s) in enumerate(zip(grad.shape, shape)):
            if s == 1:
                grad = grad.sum(axis=i, keepdims=True)
        return grad

In [20]:
class Neuron:
    def __init__(self, nin, nonlin = True):
        self.w = [Tensor(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Tensor(0.0)
        self.nonlin = nonlin

    def __call__(self,x):
        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons =  [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self,x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nout):
        sizes = [nin] + nout
        self.layers =[Layer(sizes[i], sizes[i+1], nonlin = (i != len(nout)-1)) for i in range(len(nout))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x)==1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [21]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    nodes, edges = trace(root)

    for n in nodes:
        # A rectangular node for every Value — shows data and grad
        dot.node(
            name=str(id(n)),
            label="{ data: %.4f | grad: %.4f }" % (n.data, n.grad),
            shape='record'
        )
        if n._op:
            # A small circle node for the operation that created it
            op_id = str(id(n)) + n._op
            dot.node(name=op_id, label=n._op)
            dot.edge(op_id, str(id(n)))   # op → output value

    for n1, n2 in edges:
        # input value → op node
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [22]:
X = Tensor([[4.0, 3.0], [-2.0, -1.0], [1.0, -1.5], [-2.0, -2.0], [-3.0, -5.0], [4.0, -1.0], [7.0, 9.0], [-1.0,5.0]])
y = Tensor([[1.0], [1.0], [-1.0], [1.0], [1.0], [-1.0], [1.0], [-1.0]])

np.random.seed(56)
W1 = Tensor(np.random.randn(2, 8) * 0.1)
b1 = Tensor(np.zeros((1, 8)))
W2 = Tensor(np.random.randn(8, 1) * 0.1)
b2 = Tensor(np.zeros((1, 1)))

params = [W1, b1, W2, b2]

for step in range(500):
    h  = (X @ W1 + b1).relu()
    out = h @ W2 + b2

    loss = ((out - y) ** 2).sum()

    for p in params:
        p.grad = np.zeros_like(p.data)
    loss.backward()

    for p in params:
        p.data -= 0.001* p.grad

    print(f"step {step:2d}  loss={loss.data:.4f}")

step  0  loss=7.8282
step  1  loss=7.8282
step  2  loss=7.8282
step  3  loss=7.8282
step  4  loss=7.8282
step  5  loss=7.8282
step  6  loss=7.8282
step  7  loss=7.8282
step  8  loss=7.8282
step  9  loss=7.8282
step 10  loss=7.8282
step 11  loss=7.8282
step 12  loss=7.8282
step 13  loss=7.8282
step 14  loss=7.8282
step 15  loss=7.8282
step 16  loss=7.8282
step 17  loss=7.8282
step 18  loss=7.8282
step 19  loss=7.8282
step 20  loss=7.8282
step 21  loss=7.8282
step 22  loss=7.8282
step 23  loss=7.8282
step 24  loss=7.8282
step 25  loss=7.8282
step 26  loss=7.8282
step 27  loss=7.8282
step 28  loss=7.8282
step 29  loss=7.8282
step 30  loss=7.8282
step 31  loss=7.8282
step 32  loss=7.8282
step 33  loss=7.8282
step 34  loss=7.8282
step 35  loss=7.8282
step 36  loss=7.8282
step 37  loss=7.8282
step 38  loss=7.8282
step 39  loss=7.8282
step 40  loss=7.8282
step 41  loss=7.8282
step 42  loss=7.8282
step 43  loss=7.8282
step 44  loss=7.8282
step 45  loss=7.8282
step 46  loss=7.8282
step 47  loss

In [23]:
dot = draw_dot(loss)
dot.render('my_graph_pytorch', format='svg', cleanup=True)

TypeError: only 0-dimensional arrays can be converted to Python scalars